# Cheating Detection v3 — Plug-and-Play Notebook

Single notebook for feature extraction, training, evaluation, and error analysis.

**Quick start:**
1. Put audio folders and `{folder}GT.csv` files next to this notebook
2. Edit `TRAIN_FOLDERS` / `TEST_FOLDER` in the Config cell
3. Run all cells top to bottom — re-running is safe (skips already-processed files)

**GT file format** (`audios2GT.csv`):
```
filename,label
response_001.wav,cheating
response_002.wav,not cheating
```
Labels accepted: `cheating/not cheating`, `read/spontaneous`, `1/0`, `yes/no`

In [ ]:
# ================================================================
# CONFIGURATION -- edit this cell only
# ================================================================
from pathlib import Path

# -- Training folders (comment/uncomment to include/exclude) ------
TRAIN_FOLDERS = [
    "audios2",
    "audios4",
    # "audios6",   # <- add new batches here
]

# -- Test folder --------------------------------------------------
TEST_FOLDER = ""   # e.g. "audios5"

# -- Paths --------------------------------------------------------
NB_DIR     = Path(".").resolve()
SAVE_DIR   = NB_DIR / "checkpoints"
REVIEW_DIR = NB_DIR / "review"
SAVE_DIR.mkdir(exist_ok=True)
REVIEW_DIR.mkdir(exist_ok=True)
(REVIEW_DIR / "fp").mkdir(exist_ok=True)
(REVIEW_DIR / "fn").mkdir(exist_ok=True)

# -- Transcription ------------------------------------------------
WHISPER_MODEL  = "small"
FILLER_PROMPT  = ("Umm, let me think like, hmm... Okay here's what I'm thinking. "
                  "So uh, basically, you know, I mean, like, right.")

# -- Model ---------------------------------------------------------
W_WAVLM       = 0.6     # WavLM weight in final combination
W_TEXT         = 0.4     # Text model weight (must sum to 1.0 with W_WAVLM)
THRESHOLD      = 0.50   # Decision threshold (0.50 like old working version)
TEST_RATIO     = 0.20
RANDOM_SEED    = 42
WAVLM_MAX_SEC = 60      # Max seconds of audio loaded for WavLM

# -- Labels --------------------------------------------------------
LABEL_MAP = {
    "read":1,"Read":1,"cheating":1,"Cheating":1,"reading":1,
    "Reading":1,"yes":1,"Yes":1,"Y":1,"1":1,1:1,
    "spontaneous":0,"Spontaneous":0,"not cheating":0,"Not Cheating":0,
    "Not cheating":0,"not_cheating":0,"no":0,"No":0,"N":0,"0":0,0:0,
    "genuine":0,"Genuine":0,
}
AUDIO_EXTS = {".wav",".mp3",".m4a",".flac",".ogg",".wma",".aac",".webm",".mp4"}

assert abs(W_WAVLM + W_TEXT - 1.0) < 0.01, "Weights must sum to 1.0"
print(f"Root      : {NB_DIR}")
print(f"Train     : {TRAIN_FOLDERS}")
print(f"Test      : '{TEST_FOLDER}' or {int(TEST_RATIO*100)}% split")
print(f"Threshold : {THRESHOLD}")

## Imports & Feature Functions
Run once — defines all helpers used in later cells.

In [ ]:
import os, re, json, shutil, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm import tqdm
warnings.filterwarnings("ignore")

import xgboost as xgb
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              accuracy_score, classification_report)
from scipy.optimize import minimize_scalar

import spacy
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner","lemmatizer"])
except OSError:
    raise RuntimeError("Run: python -m spacy download en_core_web_sm")

try:
    import parselmouth
    from parselmouth.praat import call as praat_call
    HAS_PARSELMOUTH = True
    print("parselmouth: OK")
except ImportError:
    HAS_PARSELMOUTH = False
    print("parselmouth NOT installed -- jitter/shimmer/HNR will be 0.")

try:
    import torch
    from transformers import GPT2LMHeadModel, GPT2TokenizerFast
    _gpt2_tok = None; _gpt2_mdl = None
    def _gpt2():
        global _gpt2_tok, _gpt2_mdl
        if _gpt2_mdl is None:
            _gpt2_tok = GPT2TokenizerFast.from_pretrained("gpt2")
            _gpt2_mdl = GPT2LMHeadModel.from_pretrained("gpt2").eval()
        return _gpt2_mdl, _gpt2_tok
    HAS_GPT2 = True
    print("GPT-2: OK")
except ImportError:
    HAS_GPT2 = False
    print("transformers NOT available -- perplexity features will be 0.")

import librosa

# ================================================================
# CONSTANTS
# ================================================================
FILLERS = {"um","uh","uh-huh","uhm","umm","hmm","hm","er","ah","ehm","mhm"}
DISCOURSE_MARKERS = {"you know","i mean","like","basically","actually",
                     "so","well","right","okay","oh","anyway","honestly"}
HEDGES = {"i think","i guess","maybe","perhaps","probably","kind of",
          "sort of","i believe","it seems","i suppose","might be"}
SELF_REF = {"i","me","my","myself","mine","i'm","i've","i'd","i'll"}
CONTENT_POS  = {"NOUN","VERB","ADJ","ADV","PROPN"}
FUNCTION_POS = {"DET","ADP","CONJ","CCONJ","SCONJ","PRON","AUX","PART"}

TEXT_FEATURES = [
    "filler_rate","filler_count","repetition_rate","repair_rate",
    "ttr","mattr","complex_word_rate","avg_word_length",
    "n_words","n_unique_words","avg_sentence_length","std_sentence_length",
    "fragment_rate","n_sentences","self_ref_rate","discourse_marker_rate",
    "hedge_rate","noun_rate","verb_rate","adj_rate",
]
# Removed: initial_pause, longest_pause, suspicious_gap_count, suspicious_gap_ratio
PAUSE_FEATURES = [
    "pause_mean","pause_std","pause_median","pause_skew","long_pause_rate",
    "pause_ratio","n_pauses","pause_regularity","pause_before_content_ratio",
    "pause_before_function_ratio","mid_phrase_pause_rate",
    "words_per_sec","articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean","f0_std","f0_range","f0_skew","f0_slope",
    "energy_mean","energy_std","speaking_rate_std",
]
VOICE_QUALITY_FEATURES = ["jitter_local","shimmer_local","hnr_mean"]
PERPLEXITY_FEATURES    = ["mean_perplexity","burstiness"]
ALL_TEXT_COLS = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES + VOICE_QUALITY_FEATURES + PERPLEXITY_FEATURES

# ================================================================
# FEATURE FUNCTIONS
# ================================================================

def _syllable_count(word):
    word = word.lower().strip()
    if len(word) <= 3: return 1
    count, prev_vowel = 0, False
    for ch in word:
        iv = ch in "aeiouy"
        if iv and not prev_vowel: count += 1
        prev_vowel = iv
    if word.endswith("e") and count > 1: count -= 1
    return max(count, 1)

def _mattr(words, window=50):
    if len(words) < window: return len(set(words)) / max(len(words), 1)
    return np.mean([len(set(words[i:i+window])) / window for i in range(len(words)-window+1)])

def compute_text_features(text):
    if not text or len(text.strip()) < 10: return {k:0 for k in TEXT_FEATURES}
    text_lower = text.lower().strip()
    doc = nlp(text_lower)
    words = [t.text for t in doc if t.is_alpha]
    n_words = len(words)
    if n_words < 5: return {k:0 for k in TEXT_FEATURES}

    all_toks = [t.text for t in doc]
    filler_count = sum(1 for w in all_toks if w in FILLERS)
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    bc = Counter(bigrams)
    rep_rate = sum(c-1 for c in bc.values() if c>1) / max(len(bigrams),1)
    repairs = ["i mean","no wait","sorry i","actually no","wait no","no no"]
    sentences = list(doc.sents)
    n_sents = max(len(sentences), 1)
    repair_c = sum(text_lower.count(m) for m in repairs)
    sl = [len([t for t in s if t.is_alpha]) for s in sentences]
    pos_c = Counter(t.pos_ for t in doc)
    tp = sum(pos_c.values()) or 1
    return {
        "filler_rate":       round(filler_count/n_words, 4),
        "filler_count":      filler_count,
        "repetition_rate":   round(rep_rate, 4),
        "repair_rate":       round(repair_c/n_sents, 4),
        "ttr":               round(len(set(words))/n_words, 4),
        "mattr":             round(_mattr(words), 4),
        "complex_word_rate": round(sum(1 for w in words if _syllable_count(w)>=3)/n_words, 4),
        "avg_word_length":   round(np.mean([len(w) for w in words]), 2),
        "n_words":           n_words,
        "n_unique_words":    len(set(words)),
        "avg_sentence_length": round(np.mean(sl), 2) if sl else 0,
        "std_sentence_length": round(np.std(sl), 2)  if len(sl)>1 else 0,
        "fragment_rate":     round(sum(1 for x in sl if x<4)/n_sents, 4),
        "n_sentences":       n_sents,
        "self_ref_rate":     round(sum(1 for w in all_toks if w in SELF_REF)/n_words, 4),
        "discourse_marker_rate": round(sum(text_lower.count(d) for d in DISCOURSE_MARKERS)/n_sents, 4),
        "hedge_rate":        round(sum(text_lower.count(h) for h in HEDGES)/n_sents, 4),
        "noun_rate":         round(pos_c.get("NOUN",0)/tp, 4),
        "verb_rate":         round(pos_c.get("VERB",0)/tp, 4),
        "adj_rate":          round(pos_c.get("ADJ",0)/tp, 4),
    }

def compute_pause_features(words):
    empty = {k:0 for k in PAUSE_FEATURES}
    if not words or len(words) < 5: return empty
    pauses = []
    for i in range(1, len(words)):
        gap = words[i]["start"] - words[i-1]["end"]
        if gap > 0.05:
            pauses.append({"dur": gap, "after_word": words[i-1].get("word",""),
                           "before_word": words[i].get("word",""), "pos": i})

    if not pauses:
        return {k:0 for k in PAUSE_FEATURES}

    durs = [p["dur"] for p in pauses]
    total_dur = words[-1]["end"] - words[0]["start"]
    speaking_dur = total_dur - sum(durs)
    doc = nlp(" ".join(w.get("word","") for w in words))
    tok_pos = {t.text.lower(): t.pos_ for t in doc}
    nbc, nbf, nmp = 0, 0, 0
    for p in pauses:
        pos = tok_pos.get(p["before_word"].lower().strip(".,!?"), "X")
        if pos in CONTENT_POS:  nbc += 1
        elif pos in FUNCTION_POS: nbf += 1
        if not p["after_word"].endswith((".",",","!","?")): nmp += 1
    n_p = len(pauses)
    positions = [p["pos"] for p in pauses]
    intervals = [positions[i]-positions[i-1] for i in range(1,len(positions))]

    return {
        "pause_mean":      round(np.mean(durs),4),
        "pause_std":       round(np.std(durs),4),
        "pause_median":    round(float(np.median(durs)),4),
        "pause_skew":      round(float(pd.Series(durs).skew()) if len(durs)>2 else 0,4),
        "long_pause_rate": round(sum(1 for d in durs if d>0.5)/n_p,4),
        "pause_ratio":     round(sum(durs)/max(total_dur,0.1),4),
        "n_pauses":        n_p,
        "pause_regularity": round(np.std(intervals) if intervals else 0,4),
        "pause_before_content_ratio":  round(nbc/n_p,4),
        "pause_before_function_ratio": round(nbf/n_p,4),
        "mid_phrase_pause_rate":       round(nmp/n_p,4),
        "words_per_sec":   round(len(words)/max(total_dur,0.1),2),
        "articulation_rate": round(len(words)/max(speaking_dur,0.1),2),
    }

def compute_prosodic_features(audio_path):
    empty = {k:0 for k in PROSODIC_FEATURES}
    try:
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True, duration=120)
    except Exception: return empty
    if len(audio) < 16000: return empty
    f0, _, _ = librosa.pyin(audio, fmin=75, fmax=500, sr=16000, frame_length=2048)
    fv = f0[~np.isnan(f0)] if f0 is not None else np.array([])
    if len(fv) < 10: return empty
    slope = float(np.polyfit(np.arange(len(fv)), fv, 1)[0])
    rms = librosa.feature.rms(y=audio, frame_length=512, hop_length=256)[0]
    win = 2*16000
    rates = [float((librosa.feature.rms(y=audio[s:s+win], frame_length=512, hop_length=256)[0]
                    > np.percentile(rms,20)).mean())
             for s in range(0, len(audio)-win, 16000)]
    return {
        "f0_mean":  round(float(np.mean(fv)),2), "f0_std": round(float(np.std(fv)),2),
        "f0_range": round(float(fv.max()-fv.min()),2), "f0_skew": round(float(pd.Series(fv).skew()),4),
        "f0_slope": round(slope,6), "energy_mean": round(float(np.mean(rms)),6),
        "energy_std": round(float(np.std(rms)),6),
        "speaking_rate_std": round(float(np.std(rates)) if rates else 0,4),
    }

def compute_voice_quality(audio_path):
    empty = {"jitter_local":0,"shimmer_local":0,"hnr_mean":0}
    if not HAS_PARSELMOUTH: return empty
    try:
        snd  = parselmouth.Sound(str(audio_path))
        pp   = praat_call(snd, "To PointProcess (periodic, cc)", 75, 500)
        jit  = praat_call(pp,  "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        shim = praat_call([snd, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        harm = praat_call(snd, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr  = praat_call(harm, "Get mean", 0, 0)
        return {"jitter_local": round(float(jit),6),
                "shimmer_local": round(float(shim),6),
                "hnr_mean": round(float(hnr),4)}
    except Exception: return empty

def compute_perplexity_features(text):
    empty = {"mean_perplexity":0,"burstiness":0}
    if not HAS_GPT2 or not text or len(text.strip()) < 20: return empty
    try:
        mdl, tok = _gpt2()
        sents = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if len(s.split())>3]
        if not sents: return empty
        ppls = []
        for s in sents[:20]:
            enc = tok(s, return_tensors="pt", truncation=True, max_length=256)
            with torch.no_grad():
                loss = mdl(**enc, labels=enc["input_ids"]).loss
            ppls.append(float(torch.exp(loss)))
        return {"mean_perplexity": round(float(np.mean(ppls)),2),
                "burstiness":      round(float(np.var(ppls)),2)}
    except Exception: return empty

print("All feature functions loaded.")
print(f"Total text model features: {len(ALL_TEXT_COLS)}")

## 1. Scan Folders
Finds audio files and checks which features already exist.

In [ ]:
def scan_folder(folder_name):
    audio_dir = NB_DIR / folder_name
    if not audio_dir.exists():
        print(f"  SKIP {folder_name}: folder not found")
        return None
    audio_files = sorted(f for f in audio_dir.rglob("*")
                         if f.suffix.lower() in AUDIO_EXTS and f.is_file())
    if not audio_files:
        print(f"  SKIP {folder_name}: no audio files")
        return None
    gt_path = NB_DIR / f"{folder_name}GT.csv"
    if not gt_path.exists():
        print(f"  SKIP {folder_name}: no GT file ({folder_name}GT.csv not found)")
        return None
    t_json  = NB_DIR / f"{folder_name}_transcripts.json"
    f_csv   = NB_DIR / f"{folder_name}_features.csv"
    wl_csv  = NB_DIR / f"{folder_name}_wavlm.csv"
    done_t  = len(json.load(open(t_json,encoding="utf-8"))) if t_json.exists() else 0
    done_f  = len(pd.read_csv(f_csv)) if f_csv.exists() else 0
    done_w  = len(pd.read_csv(wl_csv)) if wl_csv.exists() else 0
    n       = len(audio_files)
    print(f"  {folder_name}: {n} files | transcripts {done_t}/{n} | features {done_f}/{n} | wavlm {done_w}/{n}")
    return {"name":folder_name,"audio_dir":audio_dir,"audio_files":audio_files,
            "gt_path":gt_path,"t_json":t_json,"f_csv":f_csv,"wl_csv":wl_csv}

print("=== TRAINING FOLDERS ===")
train_meta = [m for fn in TRAIN_FOLDERS for m in [scan_folder(fn)] if m]

test_meta = None
if TEST_FOLDER:
    print("\n=== TEST FOLDER ===")
    test_meta = scan_folder(TEST_FOLDER)

all_meta = train_meta + ([test_meta] if test_meta else [])
print(f"\nReady: {len(train_meta)} train folder(s)", ("+ 1 test folder" if test_meta else ""))

## 2. Transcribe Audio (resume-safe)
Skips files already in the JSON. Uses Whisper with filler prompt.

In [ ]:
from faster_whisper import WhisperModel

print(f"Loading Whisper {WHISPER_MODEL}...")
_whisper = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
print("Whisper loaded.")

def transcribe_folder(meta):
    t_json = meta["t_json"]
    existing = json.load(open(t_json, encoding="utf-8")) if t_json.exists() else {}
    todo = [f for f in meta["audio_files"] if f.name not in existing]
    if not todo:
        print(f"  {meta['name']}: all {len(existing)} files already transcribed.")
        return
    print(f"  {meta['name']}: transcribing {len(todo)} new files...")
    for fp in tqdm(todo, desc=meta["name"]):
        try:
            segs, info = _whisper.transcribe(
                str(fp), language="en", word_timestamps=True,
                initial_prompt=FILLER_PROMPT,
                vad_filter=True, vad_parameters={"min_silence_duration_ms": 100}
            )
            words, text_parts = [], []
            for seg in segs:
                text_parts.append(seg.text)
                if seg.words:
                    for w in seg.words:
                        words.append({"word": w.word.strip(), "start": round(w.start,3),
                                      "end": round(w.end,3)})
            existing[fp.name] = {"text": " ".join(text_parts).strip(), "words": words,
                                  "duration_sec": round(info.duration, 2)}
        except Exception as e:
            print(f"    FAILED {fp.name}: {e}")
            existing[fp.name] = {"text": "", "words": [], "duration_sec": 0}
        # Save after each file (resume-safe)
        with open(t_json, "w", encoding="utf-8") as fh:
            json.dump(existing, fh, ensure_ascii=False, indent=1)
    print(f"  {meta['name']}: done. Total {len(existing)} transcripts.")

for meta in all_meta:
    transcribe_folder(meta)
print("\nTranscription complete.")

## 3. Extract Features (resume-safe)
Text + pause + prosodic + voice quality (jitter/shimmer/HNR) + GPT-2 perplexity.

In [ ]:
def extract_features_folder(meta):
    f_csv = meta["f_csv"]
    t_json = meta["t_json"]
    if not t_json.exists():
        print(f"  {meta['name']}: no transcripts -- run transcription first."); return

    transcripts = json.load(open(t_json, encoding="utf-8"))

    # Check existing rows AND columns
    if f_csv.exists():
        existing_df = pd.read_csv(f_csv)
        existing = set(existing_df["filename"].tolist())
        expected_cols = set(TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES +
                           VOICE_QUALITY_FEATURES + PERPLEXITY_FEATURES)
        missing_cols = expected_cols - set(existing_df.columns)
        if missing_cols:
            print(f"  {meta['name']}: CSV missing columns {missing_cols} -- recomputing all rows")
            existing = set()  # force full recompute
    else:
        existing = set()

    todo = [fp for fp in meta["audio_files"] if fp.name not in existing]
    if not todo:
        print(f"  {meta['name']}: all features already extracted."); return

    print(f"  {meta['name']}: extracting {len(todo)} new files...")
    rows = []
    for fp in tqdm(todo, desc=meta["name"]):
        t = transcripts.get(fp.name, {"text":"","words":[],"duration_sec":0})
        text, words = t["text"], t["words"]

        tf  = compute_text_features(text)
        pf  = compute_pause_features(words)
        pro = compute_prosodic_features(fp)
        vq  = compute_voice_quality(fp)
        ppx = compute_perplexity_features(text)

        row = {"filename": fp.name, "duration_sec": t.get("duration_sec",0),
               "n_words_asr": len(words), "transcript": text[:300]}
        row.update(tf); row.update(pf); row.update(pro); row.update(vq); row.update(ppx)
        rows.append(row)

        if len(rows) % 10 == 0:
            chunk = pd.DataFrame(rows)
            chunk.to_csv(f_csv, mode="a", header=not f_csv.exists(), index=False)
            rows = []

    if rows:
        chunk = pd.DataFrame(rows)
        chunk.to_csv(f_csv, mode="a", header=not f_csv.exists(), index=False)
    print(f"  {meta['name']}: done.")

for meta in all_meta:
    extract_features_folder(meta)
print("\nFeature extraction complete.")

## 4. Extract WavLM Embeddings (resume-safe)
768-dim mean-pooled encoder embeddings from WavLM-base-plus.

In [ ]:
import torch
from transformers import AutoFeatureExtractor, WavLMModel

print("Loading WavLM-base-plus (downloads on first run ~360MB)...")
_wlm_fe  = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
_wlm_mdl = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").eval()
WAVLM_DIM = _wlm_mdl.config.hidden_size   # 768
print(f"WavLM loaded. Embedding dim: {WAVLM_DIM}")

def extract_wavlm_folder(meta):
    wl_csv = meta["wl_csv"]
    existing = set(pd.read_csv(wl_csv)["filename"].tolist()) if wl_csv.exists() else set()
    todo = [fp for fp in meta["audio_files"] if fp.name not in existing]
    if not todo:
        print(f"  {meta['name']}: all WavLM embeddings already extracted."); return

    print(f"  {meta['name']}: extracting {len(todo)} embeddings...")
    rows = []
    for fp in tqdm(todo, desc=meta["name"]):
        try:
            audio, _ = librosa.load(str(fp), sr=16000, mono=True, duration=WAVLM_MAX_SEC)
            if len(audio) < 16000:
                emb = np.zeros(WAVLM_DIM)
            else:
                with torch.no_grad():
                    inp = _wlm_fe(audio, sampling_rate=16000, return_tensors="pt", padding=True)
                    out = _wlm_mdl(**inp)
                    emb = out.last_hidden_state.mean(dim=1).squeeze().numpy()
        except Exception as e:
            print(f"    FAILED {fp.name}: {e}"); emb = np.zeros(WAVLM_DIM)

        row = {"filename": fp.name}
        row.update({f"wavlm_{i}": round(float(emb[i]),6) for i in range(WAVLM_DIM)})
        rows.append(row)

        if len(rows) % 20 == 0:
            chunk = pd.DataFrame(rows)
            chunk.to_csv(wl_csv, mode="a", header=not wl_csv.exists(), index=False)
            rows = []

    if rows:
        chunk = pd.DataFrame(rows)
        chunk.to_csv(wl_csv, mode="a", header=not wl_csv.exists(), index=False)
    print(f"  {meta['name']}: done.")

for meta in all_meta:
    extract_wavlm_folder(meta)
print("\nWavLM extraction complete.")

## 5. Load & Combine Data
Merges features with GT labels. Builds train/test dataframes.

In [ ]:
def load_folder_data(meta):
    gt = pd.read_csv(meta["gt_path"])
    fname_col = "filename" if "filename" in gt.columns else gt.columns[0]
    label_col = next((c for c in gt.columns if c.lower() in
                      ["label","label_int","gt","ground_truth","cheating"]), gt.columns[-1])
    gt["label_int"] = gt[label_col].map(lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x), -1)))
    gt_map = dict(zip(gt[fname_col], gt["label_int"]))

    feat = pd.read_csv(meta["f_csv"])
    feat["label_int"] = feat["filename"].map(gt_map).fillna(-1).astype(int)
    feat["batch"] = meta["name"]
    feat = feat[feat["label_int"].isin([0,1])].reset_index(drop=True)

    wl = pd.read_csv(meta["wl_csv"])
    wl["label_int"] = wl["filename"].map(gt_map).fillna(-1).astype(int)
    wl["batch"] = meta["name"]
    wl = wl[wl["label_int"].isin([0,1])].reset_index(drop=True)

    print(f"  {meta['name']}: {len(feat)} labeled samples | "
          f"cheating={feat['label_int'].sum()} | not={(feat['label_int']==0).sum()}")
    return feat, wl

print("=== TRAINING DATA ===")
train_feat_dfs, train_wl_dfs = [], []
for meta in train_meta:
    if not meta["f_csv"].exists() or not meta["wl_csv"].exists():
        print(f"  SKIP {meta['name']}: features not extracted yet"); continue
    tf, tw = load_folder_data(meta)
    train_feat_dfs.append(tf); train_wl_dfs.append(tw)

train_feat = pd.concat(train_feat_dfs, ignore_index=True)
train_wl   = pd.concat(train_wl_dfs,   ignore_index=True)
train_wl   = train_wl.set_index("filename").reindex(train_feat["filename"]).reset_index()
groups     = train_feat["batch"].values
y_all      = train_feat["label_int"].values

# Only use features in ALL_TEXT_COLS (excludes suspicious features)
text_cols  = [c for c in ALL_TEXT_COLS if c in train_feat.columns]
# Use ALL 768 WavLM dims directly -- NO PCA
wavlm_cols = [c for c in train_wl.columns if c.startswith("wavlm_")]
X_text_all = train_feat[text_cols].fillna(0).values
X_wl_all   = train_wl[wavlm_cols].fillna(0).values
print(f"\nTotal train: {len(train_feat)} samples | text features: {len(text_cols)} | wavlm: {len(wavlm_cols)}")
print(f"Cheating: {y_all.sum()} | Not: {(y_all==0).sum()} | Batches: {sorted(set(groups))}")

has_explicit_test = test_meta is not None and test_meta["f_csv"].exists() and test_meta["wl_csv"].exists()
if has_explicit_test:
    print("\n=== TEST DATA ===")
    test_feat, test_wl = load_folder_data(test_meta)
    test_wl = test_wl.set_index("filename").reindex(test_feat["filename"]).reset_index()
else:
    print("\n=== TEST = 20% SPLIT FROM TRAIN ===")
    idx = np.arange(len(train_feat))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_RATIO, random_state=RANDOM_SEED,
                                       stratify=y_all)
    test_feat = train_feat.iloc[idx_te].reset_index(drop=True)
    test_wl   = train_wl.iloc[idx_te].reset_index(drop=True)
    train_feat = train_feat.iloc[idx_tr].reset_index(drop=True)
    train_wl   = train_wl.iloc[idx_tr].reset_index(drop=True)
    groups     = groups[idx_tr]
    y_all      = y_all[idx_tr]
    X_text_all = X_text_all[idx_tr]
    X_wl_all   = X_wl_all[idx_tr]
    print(f"  Train: {len(train_feat)} | Test: {len(test_feat)}")

y_test  = test_feat["label_int"].values
X_text_test_raw = test_feat[text_cols].fillna(0).values
X_wl_test_raw   = test_wl[wavlm_cols].fillna(0).values
print("Data ready.")

## 6. Train Models
Trains two XGBoost models on all training data, then calibrates via temperature scaling.

- **Text XGBoost** — 48 handcrafted features (text + pause + prosodic + voice quality + perplexity)
- **WavLM XGBoost** — WavLM 768-dim → PCA 80-dim (reduces overfitting)
- **Final score** — calibrated weighted average: `W_WAVLM * wavlm_prob + W_TEXT * text_prob`

In [ ]:
from scipy.optimize import minimize_scalar

np.random.seed(RANDOM_SEED)

def temperature_scale(proba_train, y_train, proba_val=None):
    eps = 1e-7
    logits = np.log(np.clip(proba_train, eps, 1-eps) / np.clip(1-proba_train, eps, 1-eps))
    def nll(T):
        p = 1/(1+np.exp(-logits/T))
        return -np.mean(y_train*np.log(p+eps)+(1-y_train)*np.log(1-p+eps))
    T = minimize_scalar(nll, bounds=(0.1,5.0), method="bounded").x
    if proba_val is None: return T
    logits_v = np.log(np.clip(proba_val,eps,1-eps)/np.clip(1-proba_val,eps,1-eps))
    return 1/(1+np.exp(-logits_v/T)), T

print("=== 5-fold Cross-Validation (GroupKFold by batch) ===")
n_splits = min(5, len(set(groups)))
gkf = GroupKFold(n_splits=n_splits)
oof_text = np.zeros(len(y_all))
oof_wl   = np.zeros(len(y_all))

for fold, (tr, va) in enumerate(gkf.split(X_text_all, y_all, groups)):
    # Text model
    sc_t = StandardScaler().fit(X_text_all[tr])
    spw  = (y_all[tr]==0).sum() / max((y_all[tr]==1).sum(),1)
    m_t  = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                               scale_pos_weight=spw, eval_metric="logloss",
                               early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_t.fit(sc_t.transform(X_text_all[tr]), y_all[tr],
            eval_set=[(sc_t.transform(X_text_all[va]), y_all[va])], verbose=False)
    raw_t = m_t.predict_proba(sc_t.transform(X_text_all[va]))[:,1]
    cal_t, _ = temperature_scale(m_t.predict_proba(sc_t.transform(X_text_all[tr]))[:,1],
                                  y_all[tr], raw_t)
    oof_text[va] = cal_t

    # WavLM model -- NO PCA, raw 768 dims, colsample_bytree=0.2
    sc_w  = StandardScaler().fit(X_wl_all[tr])
    X_wl_tr = sc_w.transform(X_wl_all[tr])
    X_wl_va = sc_w.transform(X_wl_all[va])
    m_w  = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.2, min_child_weight=3,
                               scale_pos_weight=spw, eval_metric="logloss",
                               early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_w.fit(X_wl_tr, y_all[tr], eval_set=[(X_wl_va, y_all[va])], verbose=False)
    raw_w = m_w.predict_proba(X_wl_va)[:,1]
    cal_w, _ = temperature_scale(m_w.predict_proba(X_wl_tr)[:,1], y_all[tr], raw_w)
    oof_wl[va] = cal_w

oof_combined = W_WAVLM * oof_wl + W_TEXT * oof_text
for name, oof in [("Text",oof_text),("WavLM",oof_wl),("Combined",oof_combined)]:
    preds = (oof >= THRESHOLD).astype(int)
    print(f"  {name:12s}  prec={precision_score(y_all,preds,zero_division=0):.3f}  "
          f"rec={recall_score(y_all,preds,zero_division=0):.3f}  "
          f"f1={f1_score(y_all,preds,zero_division=0):.3f}")

print("\n=== Training final models on all training data ===")
spw_final = (y_all==0).sum() / max((y_all==1).sum(),1)

text_scaler = StandardScaler().fit(X_text_all)
text_model  = xgb.XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                                  scale_pos_weight=spw_final, eval_metric="logloss",
                                  random_state=RANDOM_SEED)
text_model.fit(text_scaler.transform(X_text_all), y_all, verbose=False)
T_text = temperature_scale(text_model.predict_proba(text_scaler.transform(X_text_all))[:,1], y_all)

# WavLM: scale only, NO PCA
wl_scaler = StandardScaler().fit(X_wl_all)
X_wl_scaled = wl_scaler.transform(X_wl_all)
wl_model  = xgb.XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.2, min_child_weight=3,
                                scale_pos_weight=spw_final, eval_metric="logloss",
                                random_state=RANDOM_SEED)
wl_model.fit(X_wl_scaled, y_all, verbose=False)
T_wl = temperature_scale(wl_model.predict_proba(X_wl_scaled)[:,1], y_all)

print(f"Text model: temperature T={T_text:.3f}")
print(f"WavLM model: temperature T={T_wl:.3f}")
print(f"WavLM features: {X_wl_all.shape[1]} dims (raw, no PCA)")
print("Models trained.")

## 7. Evaluate on Test Set

In [ ]:
def calibrated_predict(X_text_raw, X_wl_raw):
    eps = 1e-7
    # Text
    p_t  = text_model.predict_proba(text_scaler.transform(X_text_raw))[:,1]
    lg_t = np.log(np.clip(p_t,eps,1-eps)/np.clip(1-p_t,eps,1-eps))
    p_t_cal = 1/(1+np.exp(-lg_t/T_text))
    # WavLM -- scale only, no PCA
    p_w  = wl_model.predict_proba(wl_scaler.transform(X_wl_raw))[:,1]
    lg_w = np.log(np.clip(p_w,eps,1-eps)/np.clip(1-p_w,eps,1-eps))
    p_w_cal = 1/(1+np.exp(-lg_w/T_wl))
    combined = W_WAVLM * p_w_cal + W_TEXT * p_t_cal
    return p_t_cal, p_w_cal, combined

p_text_test, p_wl_test, p_combined_test = calibrated_predict(X_text_test_raw, X_wl_test_raw)
preds_test = (p_combined_test >= THRESHOLD).astype(int)

print(f"=== TEST SET RESULTS (threshold={THRESHOLD}) ===")
print(f"Samples: {len(y_test)} | Cheating: {y_test.sum()} | Not: {(y_test==0).sum()}")
print()
for name, p in [("Text",p_text_test),("WavLM",p_wl_test),("Combined",p_combined_test)]:
    pr = (p >= THRESHOLD).astype(int)
    print(f"  {name:12s}  prec={precision_score(y_test,pr,zero_division=0):.3f}  "
          f"rec={recall_score(y_test,pr,zero_division=0):.3f}  "
          f"f1={f1_score(y_test,pr,zero_division=0):.3f}")
print()
print(classification_report(y_test, preds_test, target_names=["not cheating","cheating"]))

## 8. Threshold Analysis
Sweep thresholds to understand precision/recall trade-off.

In [ ]:
thresholds = np.arange(0.25, 0.81, 0.05)
rows = []
for thr in thresholds:
    pr = (p_combined_test >= thr).astype(int)
    n_pred = pr.sum()
    rows.append({
        "threshold": round(thr,2),
        "predicted_cheating": n_pred,
        "precision": round(precision_score(y_test, pr, zero_division=0),3),
        "recall":    round(recall_score(y_test, pr, zero_division=0),3),
        "f1":        round(f1_score(y_test, pr, zero_division=0),3),
        "fp": int(((pr==1)&(y_test==0)).sum()),
        "fn": int(((pr==0)&(y_test==1)).sum()),
    })

thresh_df = pd.DataFrame(rows)
print("=== THRESHOLD SWEEP (Combined Score) ===")
print(thresh_df.to_string(index=False))

# Best precision >= 0.90 with max recall
high_prec = thresh_df[thresh_df["precision"] >= 0.90]
if len(high_prec):
    best = high_prec.loc[high_prec["recall"].idxmax()]
    print(f"\nBest at prec>=0.90: threshold={best['threshold']} | "
          f"prec={best['precision']} rec={best['recall']} f1={best['f1']}")
else:
    print("\nNo configuration achieves precision >= 0.90 on this test set.")

thresh_df.to_csv(NB_DIR / "threshold_analysis.csv", index=False)
print("Saved: threshold_analysis.csv")

## 9. Save Predictions

In [ ]:
results = test_feat[["filename"]].copy()
results["text_score"]     = np.round(p_text_test, 4)
results["wavlm_score"]    = np.round(p_wl_test, 4)
results["combined_score"] = np.round(p_combined_test, 4)
results["prediction"]     = preds_test
results["prediction_str"] = results["prediction"].map({1:"cheating", 0:"not cheating"})
if "label_int" in test_feat.columns:
    results["true_label"]     = test_feat["label_int"].values
    results["true_label_str"] = results["true_label"].map({1:"cheating", 0:"not cheating"})
    results["correct"]        = (results["prediction"] == results["true_label"]).astype(int)

out_csv = NB_DIR / "predictions.csv"
results.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
print(results.head(10).to_string(index=False))

## 10. Error Analysis
Shows false positives (FP) and false negatives (FN), saves a CSV, and **copies** misclassified audio files to `review/fp/` and `review/fn/`.

In [ ]:
if "true_label" not in results.columns:
    print("No GT labels available — skip error analysis."); raise SystemExit(0)

fp_df = results[(results["prediction"]==1) & (results["true_label"]==0)].copy()
fn_df = results[(results["prediction"]==0) & (results["true_label"]==1)].copy()
tp_df = results[(results["prediction"]==1) & (results["true_label"]==1)].copy()
tn_df = results[(results["prediction"]==0) & (results["true_label"]==0)].copy()

print(f"TP={len(tp_df)}  TN={len(tn_df)}  FP={len(fp_df)}  FN={len(fn_df)}")
print(f"Precision={len(tp_df)/max(len(tp_df)+len(fp_df),1):.3f}  "
      f"Recall={len(tp_df)/max(len(tp_df)+len(fn_df),1):.3f}")

# ── Save error CSV ────────────────────────────────────────────────
errors = pd.concat([fp_df.assign(error_type="FP"), fn_df.assign(error_type="FN")])
errors = errors.sort_values("combined_score", ascending=False)
errors.to_csv(NB_DIR / "errors.csv", index=False)
print(f"\nSaved: errors.csv ({len(errors)} misclassified samples)")

# ── Copy audio files to review/ folders ──────────────────────────
# Build filename → audio path map across all folders
audio_map = {}
for meta in all_meta:
    for fp in meta["audio_files"]:
        audio_map[fp.name] = fp

def copy_files(df, subdir, label):
    dst_dir = REVIEW_DIR / subdir
    dst_dir.mkdir(exist_ok=True)
    copied, missing = 0, 0
    for _, row in df.iterrows():
        src = audio_map.get(row["filename"])
        if src and src.exists():
            shutil.copy2(src, dst_dir / src.name)
            copied += 1
        else:
            missing += 1
    print(f"  {label}: {copied} files copied to review/{subdir}/ ({missing} missing)")

copy_files(fp_df, "fp", "FP (predicted cheating, actually not)")
copy_files(fn_df, "fn", "FN (predicted not cheating, actually cheating)")

# ── Score distributions ───────────────────────────────────────────
print("\n=== Score distribution by outcome ===")
for label, df in [("TP",tp_df),("TN",tn_df),("FP",fp_df),("FN",fn_df)]:
    if len(df):
        print(f"  {label} ({len(df):3d}): combined score mean={df['combined_score'].mean():.3f}  "
              f"std={df['combined_score'].std():.3f}  "
              f"min={df['combined_score'].min():.3f}  max={df['combined_score'].max():.3f}")

if len(fp_df):
    print("\n=== Top False Positives (highest confidence wrong predictions) ===")
    print(fp_df[["filename","combined_score","text_score","wavlm_score"]].head(10).to_string(index=False))
if len(fn_df):
    print("\n=== Top False Negatives (most missed cheaters) ===")
    print(fn_df[["filename","combined_score","text_score","wavlm_score"]].head(10).to_string(index=False))

## 11. Save Models
Saves everything needed for inference.

In [ ]:
import json as _json

joblib.dump(text_model,  SAVE_DIR / "text_model.pkl")
joblib.dump(wl_model,    SAVE_DIR / "wl_model.pkl")
joblib.dump(text_scaler, SAVE_DIR / "text_scaler.pkl")
joblib.dump(wl_scaler,   SAVE_DIR / "wl_scaler.pkl")

meta_dict = {
    "text_cols":      text_cols,
    "wavlm_cols":     wavlm_cols,
    "T_text":         float(T_text),
    "T_wl":           float(T_wl),
    "W_WAVLM":        W_WAVLM,
    "W_TEXT":          W_TEXT,
    "THRESHOLD":       THRESHOLD,
    "WAVLM_MAX_SEC":   WAVLM_MAX_SEC,
    "n_wavlm_dims":    len(wavlm_cols),
}
with open(SAVE_DIR / "meta.json", "w") as f:
    _json.dump(meta_dict, f, indent=2)

print(f"Saved to {SAVE_DIR}")